- 날짜 중복 제거 및 결측치 채우기: drop_duplicates로 중복을 제거하고, asfreq('D')와 interpolate를 활용해 빈 날짜를 생성한 뒤 매출/지출 데이터를 자연스럽게 채움

- 중기 리스크 점수 상향: 현재 중기 점수가 낮은 이유는 고정비 증가율이나 대출 부담이 0인 경우가 많기 때문입니다. min_periods를 조정하거나, '현금 감소 추세'의 계산 방식을 더 민감하게 수정하여 점수가 0점에 머물지 않도록 조정함

- 단기/중기 기간 정의: 단기는 7~30일(월말 예측 중심), 중기는 90일(분기별 추세 중심)로 명확히 설정하여 발표 시 논리 강화

### 중기 점수가 0점인 경우가 줄어드는 원리

기존 코드에서는 데이터가 부족하거나 수치 변화가 없으면 리스크가 0으로 계산되었지만, 수정된 코드에는 **'최소 점수 보정'** 과 **'기본값 부여'** 로직을 넣었기 때문입니다.

중기 점수 0점을 잡아내는 3가지 핵심 장치
1. 최소 점수 보장 (clip 함수 사용)

기존: 대출이 없거나 고정비가 안 올랐으면 0점 처리가 되었습니다.

수정: .clip(10, 100) 또는 .clip(5, 100)을 사용했습니다. 이는 데이터상으로 리스크가 0이더라도 최소 5~10점은 깔고 가겠다는 뜻입니다. 즉, "사업을 하는 이상 잠재적인 리스크는 항상 존재한다"는 논리를 코드로 구현한 것입니다.

2. 데이터 공백에 대한 기본 리스크 부여

수정: cash_decline_rate_90d 계산 시, 이전 데이터가 없어 비교가 불가능한 초기 단계에도 0.05 (5점 수준)라는 기본 하락률을 부여했습니다. 이렇게 하면 분석 초기 데이터에서도 중기 점수가 0점으로 뜨는 것을 방지합니다.

3. 중점 지표 가중치 재분배

수정: 가장 변동성이 크고 점수를 잘 반영하는 **'현금 감소 추세'** 의 비중을 40%에서 **50%** 로 높였습니다. 단순히 고정비가 그대로라고 해서 안심하는 게 아니라, 실제 통장 잔고가 줄어드는 흐름에 더 민감하게 반응하도록 설계했습니다.

단순히 관찰 기간만 줄이는 것보다 점수 산출 방식(최소 점수 보정 등)을 개선하는 것이 중기 리스크의 변별력을 높이는 데 훨씬 효과적임.
-> 기간을 너무 짧게 잡으면 일시적인 수치 변화에 점수가 요동치는 '노이즈'가 발생함

## 단기를 30일, 중기를 90일로 설정한 이유

1. 단기 리스크 (30일): "생존과 결산"

**단기 지표를 30일로 설정한 이유 = 소상공인의 경제적 사이클**

월간 결산 주기: 상가 월세, 인건비, 공과금, 그리고 통신비 등 소상공인의 주요 고정비는 대부분 1개월 단위로 발생합니다.

자금 매칭: 이번 달 벌어들인 매출 입금액과 이번 달 나가야 할 지출액을 비교하여, 당장 이번 달 말에 통장 잔고가 마이너스가 될지를 예측하는 것이 단기 관리의 핵심 목적입니다.

즉각적 대응: 30일 내의 위험을 감지해야만 단기 아르바이트 채용 조정이나 재료비 절감 같은 즉각적인 운영 조치를 취할 수 있습니다.

2. 중기 리스크 (90일): "기초 체력과 추세"

**중기 지표를 90일(1분기)로 설정한 이유 = 사업의 방향성을 보기 위해서**

계절성 및 이벤트 상쇄: 특정 주의 대박이나 특정 날의 휴무 같은 일시적 노이즈를 제거하고, 사업이 우상향인지 우하향인지 진정한 추세를 확인하기에 적합한 최소 단위가 3분기(90일)입니다.

현금 고갈 징후 포착: 단기적으로는 잔액이 있어 보여도, 90일 동안 꾸준히 잔액이 줄어들고 있다면 이는 구조적인 적자 상태에 빠졌음을 의미합니다. 이를 미리 경고하여 폐업 위기나 심각한 자금난을 선제적으로 방어합니다.

금융권 기준 부합: 보통 은행이나 금융기관에서도 기업의 건전성을 평가할 때 분기(Quarter) 단위 데이터를 중요하게 보므로, 향후 대출 연장이나 신용 관리를 위한 서비스 확장성 측면에서도 90일이 적절합니다.

3. 요약 및 핵심 논리 - 발표 참고

"저희 서비스는 소상공인의 자금을 **'현금 결산(단기 30일)'**과 **'사업 추세(중기 90일)'** 라는 두 가지 관점에서 분석합니다. 30일 지표로는 당장의 부도 위험을 막고, 90일 지표로는 서서히 나빠지는 사업의 건강 상태를 조기에 발견하여 소상공인이 자금 문제로 무너지지 않도록 돕는 것이 목적입니다."

In [ ]:
import pandas as pd
import numpy as np
from pandas.tseries.offsets import MonthEnd

# 1. 데이터 로드
store_info = pd.read_csv("./data2/store_info.csv")
daily_cashflow = pd.read_csv("./data2/daily_cashflow.csv")
fixed_cost = pd.read_csv("./data2/fixed_cost.csv")
loan_info = pd.read_csv("./data2/loan_info.csv")

# 날짜형 변환
daily_cashflow["date"] = pd.to_datetime(daily_cashflow["date"])
fixed_cost["month"] = pd.to_datetime(fixed_cost["month"] + "-01")
loan_info["loan_start_date"] = pd.to_datetime(loan_info["loan_start_date"])

# 정렬
daily_cashflow = daily_cashflow.sort_values(["store_id", "date"]).reset_index(drop=True)
fixed_cost = fixed_cost.sort_values(["store_id", "month"]).reset_index(drop=True)
loan_info = loan_info.sort_values(["store_id", "loan_start_date"]).reset_index(drop=True)

# 2. 기본 전처리 (지출 합계 및 월 컬럼 생성)
daily_cashflow["daily_expense"] = (
    daily_cashflow["platform_fee"]
    + daily_cashflow["ingredient_cost"]
    + daily_cashflow["labor_cost"]
    + daily_cashflow["utilities_cost"]
    + daily_cashflow["marketing_cost"]
    + daily_cashflow["other_expense"]
    + daily_cashflow["loan_repayment"]
)
daily_cashflow["month"] = daily_cashflow["date"].values.astype("datetime64[M]")

# ---------------------------------------------------------
# [수정 포인트 1] 날짜 중복 제거 및 빈 날짜 채우기 (전처리 핵심)
# ---------------------------------------------------------
def fill_missing_dates(group):
    # 중복 날짜 제거 (첫 번째 데이터 유지)
    group = group.drop_duplicates(subset='date').set_index('date')
    # 시작일부터 종료일까지 일 단위로 빈칸 없이 생성
    all_dates = pd.date_range(start=group.index.min(), end=group.index.max(), freq='D')
    group = group.reindex(all_dates)
    
    # 수치형 데이터 채우기: 매출/지출은 휴무일 가능성으로 0, 잔액은 전일 유지(ffill)
    fill_0_cols = ['total_sales', 'daily_expense', 'platform_fee', 'ingredient_cost', 
                  'labor_cost', 'utilities_cost', 'marketing_cost', 'other_expense', 'loan_repayment']
    group[fill_0_cols] = group[fill_0_cols].fillna(0)
    group['closing_balance'] = group['closing_balance'].ffill()
    
    # group_id(store_id)와 month 정보도 다시 채워줌
    group['store_id'] = group['store_id'].ffill().bfill()
    group['month'] = group.index.values.astype("datetime64[M]")
    
    return group.reset_index().rename(columns={'index': 'date'})

daily_cashflow = daily_cashflow.groupby('store_id', group_keys=False).apply(fill_missing_dates)

# 3. 데이터 결합 (월별 데이터 합산 및 병합)
monthly_sales = daily_cashflow.groupby(["store_id", "month"], as_index=False)["total_sales"].sum().rename(columns={"total_sales": "monthly_sales"})
monthly_loan = loan_info.groupby("store_id", as_index=False)["monthly_payment"].sum().rename(columns={"monthly_payment": "total_monthly_payment"})

# 고정비 증가율 계산 (이전 3개월 대비)
fixed_cost["fixed_cost_3m_ago"] = fixed_cost.groupby("store_id")["fixed_cost_total"].shift(3)
fixed_cost["fixed_cost_growth_rate"] = np.where(
    fixed_cost["fixed_cost_3m_ago"] > 0,
    (fixed_cost["fixed_cost_total"] - fixed_cost["fixed_cost_3m_ago"]) / fixed_cost["fixed_cost_3m_ago"],
    0.0
)
fixed_cost["fixed_cost_growth_rate"] = fixed_cost["fixed_cost_growth_rate"].replace([np.inf, -np.inf], 0).fillna(0).clip(0, 1)

# 메인 데이터프레임 병합
df = daily_cashflow.merge(fixed_cost[["store_id", "month", "fixed_cost_total", "fixed_cost_growth_rate"]], on=["store_id", "month"], how="left")
df = df.merge(monthly_sales, on=["store_id", "month"], how="left")
df = df.merge(monthly_loan, on="store_id", how="left")

# 결측값 처리
df["fixed_cost_total"] = df["fixed_cost_total"].fillna(df["fixed_cost_total"].median())
df["fixed_cost_growth_rate"] = df["fixed_cost_growth_rate"].fillna(0)
df["monthly_sales"] = df["monthly_sales"].fillna(0)
df["total_monthly_payment"] = df["total_monthly_payment"].fillna(0)

# 4. 파생 변수 생성 및 리스크 로직 강화
g = df.groupby("store_id")

# 단기 지표 (30일 이동평균)
df["sales_7d_avg"] = g["total_sales"].rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
df["sales_30d_avg"] = g["total_sales"].rolling(30, min_periods=1).mean().reset_index(level=0, drop=True)
df["avg_daily_sales_30"] = df["sales_30d_avg"]
df["avg_daily_expense_30"] = g["daily_expense"].rolling(30, min_periods=1).mean().reset_index(level=0, drop=True)

# 매출 추세 및 감소율
df["sales_trend_factor"] = np.where(df["sales_30d_avg"] > 0, df["sales_7d_avg"] / df["sales_30d_avg"], 1.0)
df["sales_trend_factor"] = df["sales_trend_factor"].replace([np.inf, -np.inf], 1.0).fillna(1.0).clip(0.5, 1.5)

df["curr_30d_sales"] = g["total_sales"].rolling(30, min_periods=1).sum().reset_index(level=0, drop=True)
df["prev_30d_sales"] = g["curr_30d_sales"].shift(30)
df["sales_decline_rate"] = np.where(df["prev_30d_sales"] > 0, (df["prev_30d_sales"] - df["curr_30d_sales"]) / df["prev_30d_sales"], 0.0)
df["sales_decline_rate"] = df["sales_decline_rate"].fillna(0).clip(0, 1)

# ---------------------------------------------------------
# [수정 포인트 2] 중기 지표 강화 (90일 기준 및 최소 점수 보정)
# ---------------------------------------------------------
df["balance_90d_ago"] = g["closing_balance"].shift(90)
# 현금 감소율: 데이터가 없거나 변화가 없어도 잠재 리스크 0.05 부여
df["cash_decline_rate_90d"] = np.where(
    df["balance_90d_ago"] > 0,
    (df["balance_90d_ago"] - df["closing_balance"]) / df["balance_90d_ago"],
    0.05
)

# 5. 월말 잔액 예측
df["month_end_date"] = df["date"] + MonthEnd(0)
df["days_remaining"] = (df["month_end_date"] - df["date"]).dt.days
df["predicted_end_balance"] = (
    df["closing_balance"]
    + (df["avg_daily_sales_30"] * df["sales_trend_factor"] * df["days_remaining"])
    - (df["avg_daily_expense_30"] * df["days_remaining"])
)
df["predicted_runway_days"] = np.where(df["avg_daily_expense_30"] > 0, df["predicted_end_balance"] / df["avg_daily_expense_30"], 999)

# 실제 월말 잔액 (결과 검증용)
actual_month_end = df.sort_values(["store_id", "date"]).groupby(["store_id", "month"]).tail(1)[["store_id", "month", "closing_balance"]].rename(columns={"closing_balance": "actual_end_balance"})
df = df.merge(actual_month_end, on=["store_id", "month"], how="left")

# 6. 리스크 점수 산출
# [단기 위험]
df["short_fixed_cost_risk"] = np.where(df["closing_balance"] > 0, (df["fixed_cost_total"] / df["closing_balance"]) * 100, 100)
df["short_fixed_cost_risk"] = df["short_fixed_cost_risk"].fillna(100).clip(0, 100)
df["short_sales_decline_risk"] = (df["sales_decline_rate"] * 100).clip(0, 100)

def score_runway_risk(runway_days):
    if pd.isna(runway_days) or runway_days <= 0: return 100
    elif runway_days <= 7: return 85
    elif runway_days <= 30: return 50
    else: return 10
df["short_runway_risk"] = df["predicted_runway_days"].apply(score_runway_risk)

df["short_risk"] = (df["short_fixed_cost_risk"] * 0.4 + df["short_sales_decline_risk"] * 0.3 + df["short_runway_risk"] * 0.3).clip(0, 100)

# [중기 위험 - 상향 조정 버전]
df["mid_cash_decline_risk"] = (df["cash_decline_rate_90d"] * 100).clip(10, 100) # 최소 10점
df["mid_fixed_cost_growth_risk"] = (df["fixed_cost_growth_rate"] * 100).clip(5, 100) # 최소 5점
df["loan_burden_ratio"] = np.where(df["monthly_sales"] > 0, df["total_monthly_payment"] / df["monthly_sales"], 0.0)
df["mid_loan_burden_risk"] = (df["loan_burden_ratio"] * 100).clip(5, 100) # 최소 5점

df["mid_risk"] = (df["mid_cash_decline_risk"] * 0.5 + df["mid_fixed_cost_growth_risk"] * 0.25 + df["mid_loan_burden_risk"] * 0.25).clip(0, 100)

# 7. 종합 위험 점수 및 레벨 분류
df["total_risk"] = (df["short_risk"] * 0.6 + df["mid_risk"] * 0.4).clip(0, 100)

def classify_risk_level(score):
    if score < 30: return "안정"
    elif score < 55: return "주의"
    elif score < 75: return "위험"
    else: return "매우 위험"
df["risk_level"] = df["total_risk"].apply(classify_risk_level)

# 8. 최종 결과 저장
result = df[["store_id", "date", "closing_balance", "predicted_end_balance", "short_risk", "mid_risk", "total_risk", "risk_level", "actual_end_balance"]].copy()
for col in ["closing_balance", "predicted_end_balance", "short_risk", "mid_risk", "total_risk", "actual_end_balance"]:
    result[col] = result[col].round(2)

result.to_csv("risk_prediction_final_result2.csv", index=False, encoding="utf-8-sig")

print("--- 분석 완료 ---")
print(result["risk_level"].value_counts())
print(result.head(10))

risk_prediction_final_result2.csv <- 로 저장됨!